<font color="red">**Step1** </font>  
先让我们来看看最简单的一个变量和函数类是何德何能可以实现一个微分图的呢
这都是什么意思呢？  
class Variable定义了一个简单的变量类，例如Variable(10)就会调用self.data传入自己的成员变量中，说白了就是接收10就创建一个数值为10的类型为Variable的变量。那有人可能会问这不是脱裤子放屁吗？  
class Function定义了一个函数的基函数类，输入是一个Variable类型的变量，输出是一个Variable类型的变量（其本质是np.array）  
其中使用了一个方法forward，forward本质上就是这个函数未来的具体运算，会在被继承的函数子类中重定义，正如以下的Square Exp所示  
有人可能会问，那方法为什么要被命名为forward（前向）不是我们最熟悉的fuction，这是因为在深度学习中这个由x->a->b->y的过程被叫做前向传播  
那么为什么要用类来定义这两个很基础的东西呢？  
是因为这样方便包装对象也方便以后进行其他操作。

In [2]:
import numpy as np
class Variable:
    def __init__(self,data):
        self.data = data

class Function:
    def __call__(self,input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        return output
    
    def forward(self,x):
        raise NotImplementedError()
 
class Square(Function):#一个函数的继承子类--平方函数
    def forward(self, x):
       return x**2
    
class Exp(Function):#一个函数的继承子类--指数函数
    def forward(self, x):
       return np.exp(x)


接下来，我们可以基于这样的方法和代码，生成一些较为复合复杂的函数，正如以下示例所示,y=exp(x^2)^2

In [ ]:
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.9))#--请把变量视为一个对于数的封装
a = A(x)
b = B(a)
y = C(b)##一个函数复合的构建--y=exp(x^2)^2
print(y.data)

5.0530903165638685


接下来让我们考虑对这个超级复合函数求导--根据链式法则，y对x的导数，等于y对自己的导数，乘C的导数，乘B的导数，乘A的导数  
为了完成这个工作，在我们现在的两个函数实例上，解析式求导还是简单的，X^2的导数无非就是2*X,Exp(x)的导数还是Exp(x)，当然，我们也可以考量一些更复杂的情况下，显式求导是很难完成的，只能使用数值求导。以下是一个基础的数值求导例子(前后差分).假设输入的f,x都是Variable类型的变量

In [56]:
def numerically_diff(f,x,eps = 1e-5):
    x0 = Variable(x.data - eps)
    x1 = Variable(x.data + eps)
    y0 = f(x0)
    y1 = f(x1)
    return Variable((y1.data - y0.data) / (2*eps))

def f0(x):
    return Square()(x)

def f1(x):
    A = Square()
    B = Exp()
    C = Square()
    a = A(x)
    b = B(a)
    y = C(b)
    return Variable(y).data

x=Variable(np.array(0.9))
print(numerically_diff(f0,x).data)
print(numerically_diff(f1,x).data)

1.7999999999906977
18.191125147115628


这里的f0(x)就是平方函数，f1(x)就是我们定义的复杂函数，y=exp(x^2)^2  
其中平方函数的导数我们是熟悉的，2*x吗  
0.9×2=1.8与我们数值计算的结果相差不是特别大。  
第二个显示求导的结果应该是 y'=4*x*exp(2*x^2)  
根据计算器手算 4×0.9×exp(2×0.81) = 18.191125139应该说还是基本正确的，误差在小数点后七位。  
但是问题在于  
1.这样定义函数很不方便，每次都要重新定义和调用  
2.这样对于高度复合的函数数值求导，每次都要重复封装和遍历，浪费计算复杂度。  
于是乎我们针对这些可以由一般基本函数复合成的冗长式子，做链式法则，反向求导回来，这样方便解决以上问题和封装。  
可以把这样的方法封装进backward方法中。先做下方准备工作，对于Function和Variable基类进行改进

In [14]:
class Variable:
    def __init__(self,data):
        self.data = data
        self.grad = None

首先为每一个变量添加一个标签，记录其导数。使用None初始化

In [9]:
class Function:
    def __call__(self,input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        self.input = input#新加入的，便于方向传播时重复调用输入值
        return output
    
    def forward(self,x):
        raise NotImplementedError()
    
    def backward(self,gx):#新加入的，反向传播方法
        raise NotImplementedError()
 
class Square(Function):#一个函数的继承子类--平方函数
    def forward(self, x):
       return x**2
    def backward(self,gy):#gy将会是x的上游变量y的导数，便于向下传播
        x = self.input.data
        gx = numerically_diff(self.forward, x) * gy#使用数值方法计算导数，
        return gx
    
class Exp(Function):#一个函数的继承子类--指数函数
    def forward(self, x):
       return np.exp(x)
    def backward(self,gy):
        x = self.input.data
        gx = numerically_diff(self.forward, x) * gy
        return gx

以上代码我们对我注释的部分加入了改进，其中backward方法的本质，是计算该层的导数乘上层传过来的，这样可以向下链式传递。  
接下来我们考虑没有self.input语句会带来什么后果

In [8]:
class Function:
    def __call__(self,input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        return output
    
    def forward(self,x):
        raise NotImplementedError()
    
    def backward(self,gx):#新加入的，反向传播方法
        raise NotImplementedError()
 
class Square(Function):#一个函数的继承子类--平方函数
    def forward(self, x):
       return x**2
    def backward(self,gy,x):#gy将会是x的上游变量y的导数，便于向下传播
        x = x.data#直接多加了一个变量输入
        gx = numerically_diff(self.forward, x) * gy#使用数值方法计算导数，
        return gx
    
class Exp(Function):#一个函数的继承子类--指数函数
    def forward(self, x):
       return np.exp(x)
    def backward(self,gy,x):
        x = x.data
        gx = numerically_diff(self.forward, x) * gy
        return gx

这将会使得backward需要被重新传参，这不太好。  
以及考虑到backward成员方法再各个函数里的实现大同小异，不如我们集成到某个地方。  
当然，一个朴素的想法是集成到函数基类里。  
我们之后会看到，也不是不行，但是不如集成到变量类中简单，因为一个简单的example就是第一个反向传播的时候。是以变量开始的。  
当然，我们先不考虑那么多，对这个版本的函数进行一个测试。

In [64]:
class Function:
    def __call__(self,input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        self.input = input#新加入的，便于方向传播时重复调用输入值
        return output
    
    def forward(self,x):
        raise NotImplementedError()
    
    def backward(self,gy):#新加入的，反向传播方法
        raise NotImplementedError()
 
class Square(Function):#一个函数的继承子类--平方函数
    def forward(self, x):
       return x**2
    def backward(self,gy):#gy将会是x的上游变量y的导数，便于向下传播
        x = self.input.data
        gx = 2 * x * gy#使用数值方法计算导数，并且注意类型
        return gx
    
class Exp(Function):#一个函数的继承子类--指数函数
    def forward(self, x):
       return np.exp(x)
    def backward(self,gy):
        x = self.input.data
        gx = np.exp(x) * gy
        return gx



In [65]:
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.9))#--请把变量视为一个对于数的封装
a = A(x)
b = B(a)
y = C(b)##一个函数复合的构建--y=exp(x^2)^2

y.grad = np.array(1.0)#y对自己的导数
b.grad = C.backward(y.grad)##C的反向传播
a.grad = B.backward(b.grad)##B的反向传播
x.grad = A.backward(a.grad)##A的反向传播
print(x.grad)

18.19112513962993


求导成功，但让我们的代码更加简洁可读一下。使其自动化。  
为使其自动化，我们必须重新考虑我们如何看待一个变量。现在我们的变量有两个属性，梯度和数值  
但是事实上在反向传播链条中，变量的创生函数也很重要。我们需要让我们的变量记住我们的创生函数，函数保存创造的变量的信息  
这就构建了一个计算双向链表---又叫计算图  
因为在高阶和各种其他求导下，可能会由链表衍生为图  

In [7]:
import numpy as np
class Variable:
    def __init__(self,data):
        self.data = data
        self.grad = None
        self.creator = None#新加入的，记录创建这个变量的函数

    def set_creator(self,func):#新加入的，记录创建这个变量的函数
        self.creator = func

In [8]:
class Function:
    def __call__(self,input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        output.set_creator(self)#传入self不是forward是为了保证传入整个实例
        self.input = input#新加入的，便于方向传播时重复调用输入值
        self.output = output#新加入的，记录输出变量
        return output
    
    def forward(self,x):
        raise NotImplementedError()
    
    def backward(self,gy):#新加入的，反向传播方法
        raise NotImplementedError()
 
class Square(Function):#一个函数的继承子类--平方函数
    def forward(self, x):
       return x**2
    def backward(self,gy):#gy将会是x的上游变量y的导数，便于向下传播
        x = self.input.data
        gx = 2 * x * gy#使用数值方法计算导数，并且注意类型
        return gx
    
class Exp(Function):#一个函数的继承子类--指数函数
    def forward(self, x):
       return np.exp(x)
    def backward(self,gy):
        x = self.input.data
        gx = np.exp(x) * gy
        return gx

接下来使用assert断言语句测试代码正确性。如果代码正确运行，那么不会报错也不会有事情发生。此时语句返回true

In [9]:
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.9))#--请把变量视为一个对于数的封装
a = A(x)
b = B(a)
y = C(b)


assert y.creator == C
assert y.creator.input == b
assert b.creator == B

Nothing happen尝试反向传播

In [10]:
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.9))#--请把变量视为一个对于数的封装
a = A(x)
b = B(a)
y = C(b)

y.grad = np.array(1.0)#y对自己的导数
C = y.creator#从输出变量的creator属性获得创建它的函数
b = C.input#获得C的输入变量
b.grad = C.backward(y.grad)##C的反向传播
B = b.creator#从b的creator属性获得创建它的函数
a = B.input#获得B的输入变量
a.grad = B.backward(b.grad)##B的反向传播
A = a.creator#从a的creator属性获得创建它的函数
x = A.input#获得A的输入变量
x.grad = A.backward(a.grad)##A的反向传播
print(x.grad)

18.19112513962993


接下来通过把backward方法添加进Variable实例中简化这一过程的书写。

In [11]:
class Variable:
    def __init__(self,data):
        self.data = data
        self.grad = None
        self.creator = None#新加入的，记录创建这个变量的函数

    def set_creator(self,func):#新加入的，记录创建这个变量的函数
        self.creator = func
    def backward(self):#新加入的，反向传播方法
        f = self.creator#1.获取函数
        if f is not None:
            x = f.input#2.获取函数的输入
            x.grad = f.backward(self.grad)#3.函数的反向传播
            x.backward()#4.对输入变量进行反向传播(递归)

In [12]:
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.9))#--请把变量视为一个对于数的封装
a = A(x)
b = B(a)
y = C(b)

y.grad = np.array(1.0)#y对自己的导数
y.backward()#反向传播
print(x.grad)

18.19112513962993


这下已经简洁很多了，这就是自动微分，但是递归方法，是很占用内存且运算低效的，可以该递归为循环方法。

In [19]:
class Variable:
    def __init__(self,data):
        self.data = data
        self.grad = None
        self.creator = None#新加入的，记录创建这个变量的函数

    def set_creator(self,func):#新加入的，记录创建这个变量的函数
        self.creator = func
        
    def backward(self):#新加入的，反向传播方法
        funcs = [self.creator]#函数栈
        while funcs:#函数栈不空
            f = funcs.pop()#1.弹出一个函数
            x = f.input#2.获取函数的输入
            y = f.output#新加入的，获取函数的输出
            x.grad = f.backward(y.grad)#3.函数的反向传播
            if x.creator is not None:#4.如果输入变量还存在创建函数，则将创建函数加入栈中
                funcs.append(x.creator)

其核心是每向下计算一个变量，就把其函数入栈，后再立即弹出进行反向传播，然后如此循环，避免递归调用。

下一步，我们要做这样几件事情，使我们的函数计算器更加好用
1.使函数更像一个python函数，不需要先写A=Square()
2.使backward前的y.grad = np.array(1.0)这个初始化步骤省去
3.使所有变量的类型强制保持np.array

In [ ]:
def square(x):
    f = Square()
    return f(x)

def exp(x):
    f = Exp()
    return f(x)

这一步有人可能会问那我们上面脱裤子放什么屁，最后还是要回到这样，直接**2不香吗。  
但是需要考虑到其实我们这里只是保持了表面的形式统一，其实Square的内核是一个类，远比一个计算包含了更多的信息。这样是一种更好的封装。
从运算--到复杂封装--再到简单表象，辨证美

In [20]:
x = Variable(np.array(0.9))
a = square(x)
b = exp(a)
y = square(b)

y.grad = np.array(1.0)
y.backward()
print(x.grad)

18.19112513962993


更加简洁了，接下来让我们考虑简化掉初始化。

In [22]:
class Variable:
    def __init__(self,data):
        self.data = data
        self.grad = None
        self.creator = None#新加入的，记录创建这个变量的函数

    def set_creator(self,func):#新加入的，记录创建这个变量的函数
        self.creator = func
        
    def backward(self):#新加入的，反向传播方法
        if self.grad is None:#如果grad属性没有值
            self.grad = np.ones_like(self.data)#将grad属性设置为和data属性相同形状的数组，并且元素全为1
            ###自动初始化！!


        funcs = [self.creator]#函数栈
        while funcs:#函数栈不空
            f = funcs.pop()#1.弹出一个函数
            x = f.input#2.获取函数的输入
            y = f.output#新加入的，获取函数的输出
            x.grad = f.backward(y.grad)#3.函数的反向传播
            if x.creator is not None:#4.如果输入变量还存在创建函数，则将创建函数加入栈中
                funcs.append(x.creator)

In [23]:
x = Variable(np.array(0.9))
y = square(exp(square(x)))
y.backward()
print(x.grad)

18.19112513962993


我的意思是这很好。但是考虑到用户可能会不小心使用float int类型为Variable做初始化，我们需要及时报错。

In [24]:
class Variable:
    def __init__(self,data):
        if data is not None:
            if not isinstance(data,np.ndarray):
                raise TypeError('{} is not supported'.format(type(data)))



        self.data = data
        self.grad = None
        self.creator = None#新加入的，记录创建这个变量的函数

    def set_creator(self,func):#新加入的，记录创建这个变量的函数
        self.creator = func
        
    def backward(self):#新加入的，反向传播方法
        if self.grad is None:#如果grad属性没有值
            self.grad = np.ones_like(self.data)#将grad属性设置为和data属性相同形状的数组，并且元素全为1
            ###自动初始化！!


        funcs = [self.creator]#函数栈
        while funcs:#函数栈不空
            f = funcs.pop()#1.弹出一个函数
            x = f.input#2.获取函数的输入
            y = f.output#新加入的，获取函数的输出
            x.grad = f.backward(y.grad)#3.函数的反向传播
            if x.creator is not None:#4.如果输入变量还存在创建函数，则将创建函数加入栈中
                funcs.append(x.creator)

In [25]:
x = Variable(0.9)

TypeError: <class 'float'> is not supported

将会弹出  
TypeError: <class 'float'> is not supported

由于numpy函数对于0维向量运算后，运算结果不会保持向量，会变成一个64位浮点数类型，因此我们需要一个类型转换使其类型封闭。

In [26]:
def as_array(x):
    if np.isscalar(x):##如果x是一个float输出true 则将其转换为一个numpy数组
        return np.array(x)
    return x

In [27]:
class Function:
    def __call__(self,input):
        x = input.data
        y = self.forward(x)
        output = Variable(as_array(y))
        output.set_creator(self)#传入self不是forward是为了保证传入整个实例
        self.input = input#新加入的，便于方向传播时重复调用输入值
        self.output = output#新加入的，记录输出变量
        return output
    
    def forward(self,x):
        raise NotImplementedError()
    
    def backward(self,gy):#新加入的，反向传播方法
        raise NotImplementedError()
 
class Square(Function):#一个函数的继承子类--平方函数
    def forward(self, x):
       return x**2
    def backward(self,gy):#gy将会是x的上游变量y的导数，便于向下传播
        x = self.input.data
        gx = 2 * x * gy#使用数值方法计算导数，并且注意类型
        return gx
    
class Exp(Function):#一个函数的继承子类--指数函数
    def forward(self, x):
       return np.exp(x)
    def backward(self,gy):
        x = self.input.data
        gx = np.exp(x) * gy
        return gx